# LoRA fine-tuning for interview-feedback generation (Unit 3/4 - adapting a pretrained model)

**Goal.** Adapt a small pretrained seq2seq model (Flan-T5) to write short, structured feedback for an interview answer, then compare it honestly with (a) zero-shot prompting, (b) retrieval-augmented few-shot prompting (RAG) and (c) a no-model nearest-neighbour baseline.

**What LoRA does.** Instead of updating all weights `W`, we freeze `W` and learn a low-rank update `ΔW = B·A` (rank `r` ≪ dimension) on selected projections (here the attention `q` and `v`). Only `A` and `B` are trained, so memory and time drop sharply and the adapter is a few MB.

**Honesty box.**
- Training data is `data/feedback_sft.jsonl`: **synthetic** feedback composed by `scripts/build_feedback_dataset.py` from labels and a rule-based rubric (no LLM). Every row says `reviewed: false` - a human should review (and flip the flag on) rows before you trust the model; train on reviewed rows only with `--only-reviewed`.
- Because the reference text is templated, a good score shows the model learned the **format and the grounding rules**, not that its advice teaches better than an LLM's.
- We split **by question** (test = 3 unseen questions, 51 answers). The test set is small; expect wide uncertainty.
- Run on a free Colab GPU (or CPU: `flan-t5-small` takes minutes). `PREPWISE_SMOKE=1` runs the notebook offline with a tiny *random* model: it proves the code runs, nothing more.

In [ ]:
# --- Setup: works locally (from the repo) and on Colab (clone the repo first) -------------------------
import os, sys, subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data" / "soft_skill_labeled.csv").exists():
    ROOT = ROOT.parent                       # notebook opened from notebooks/
if not (ROOT / "data" / "soft_skill_labeled.csv").exists():
    raise SystemExit("Open this notebook from the repository (or `git clone` it on Colab and cd into it).")
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
os.chdir(ROOT)
print("repo:", ROOT)

In [ ]:
!pip -q install torch transformers peft accelerate scikit-learn pandas matplotlib 2>/dev/null || true

In [ ]:
import os, json
import pandas as pd, matplotlib.pyplot as plt
import finetune_lora as F

SMOKE = os.environ.get("PREPWISE_SMOKE") == "1"      # offline wiring check only
BASE = "google/flan-t5-small"                        # try "google/flan-t5-base" on a GPU
EPOCHS = 3 if SMOKE else 10
LR = 3e-3 if SMOKE else 3e-4
OUT = ROOT / "models" / ("lora_smoke" if SMOKE else "lora_flan_t5_small")
print("SMOKE TEST (tiny random model - numbers are meaningless)" if SMOKE else f"Real run with {BASE}")

## 1. The synthetic dataset and the question-grouped split

In [ ]:
train_df, test_df = F.load_split()
print(len(train_df), "train answers /", train_df.question_id.nunique(), "questions;",
      len(test_df), "test answers /", test_df.question_id.nunique(), "UNSEEN questions;", "reviewed rows:", int(train_df.reviewed.sum()))
r = train_df.iloc[3]
print("\nINPUT :", F.to_prompt(r.question, r.answer[:160] + "..."))
print("TARGET:", r.feedback)

## 2. Base model + LoRA adapters\n`target_modules=['q','v']` are the attention query/value projections in T5; `r=8` is the adapter rank.

In [ ]:
texts = [F.to_prompt(q, a) for q, a in zip(pd.concat([train_df, test_df]).question, pd.concat([train_df, test_df]).answer)] + pd.concat([train_df, test_df]).feedback.tolist()
model, tok = F.load_base(BASE, SMOKE, texts)
if SMOKE:
    F.save_random_base(model, OUT)
base_params = sum(p.numel() for p in model.parameters())
model = F.add_lora(model, r=8, alpha=16)
summary = F.trainable_summary(model)
print(f"base parameters: {base_params:,}\nLoRA trainable:  {summary['trainable']:,} ({summary['percent']}% of the wrapped model)")

## 3. Train (plain PyTorch loop so every step is visible)

In [ ]:
losses = F.train(model, tok, train_df, epochs=EPOCHS, batch_size=8, lr=LR)
plt.plot(range(1, len(losses) + 1), losses, marker="o"); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("LoRA training loss"); plt.show()

## 4. Evaluate on unseen questions - audit the text against facts
`evaluate()` generates feedback for every test answer and checks: **format** (all four parts), **verdict** matches the label, **evidence** line is true (STAR parts found, numbers present), the criterion under **Improve** really is the weakest one, the claimed **strength** is supported, plus ROUGE-L against the reference.

In [ ]:
res = F.evaluate(model, tok, test_df)
print({k: v for k, v in res.items() if k not in ("predictions", "samples")})
for s in res["samples"][:2]:
    print(f"\n[{s['label']}] GENERATED: {s['generated']}\n         REFERENCE: {s['reference']}")

## 5. Save the adapter and reload it (round-trip check)
The adapter folder contains only the LoRA weights (+ tokenizer + `run_meta.json`).

In [ ]:
F.save(model, tok, OUT, {"base": BASE, "smoke": SMOKE, "epochs": EPOCHS, "lr": LR, "rank": 8, "seed": 42, "losses": losses,
                        "trainable": summary, "only_reviewed": False,
                        "test_metrics": {k: v for k, v in res.items() if k not in ("predictions", "samples")},
                        "split": "GroupShuffleSplit by question_id, test_frac=0.25",
                        "dataset": "data/feedback_sft.jsonl (synthetic, template-composed)"})
m2, tok2, meta = F.load_for_inference(OUT)
sub = test_df.head(4)
same = F.generate_feedback(m2, tok2, sub.question.tolist(), sub.answer.tolist()) == res["predictions"][:4]
print("reloaded adapter reproduces the generations:", same)

## 6. Prompting vs RAG vs fine-tuning (vs a no-model baseline)
`scripts/compare_approaches.py` evaluates all approaches on the **same** unseen-question test set and writes `reports/compare_approaches.md`. Configure `LLM_PROVIDER` and a key in `.env` first; with the default mock provider the two LLM rows are naive stand-ins and the report says so.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "scripts/compare_approaches.py", "--adapter", str(OUT)], capture_output=True, text=True)
print("\n".join(l[:230] for l in r.stdout.splitlines() if l.split(" ")[0] in ("nn_copy", "prompting", "rag_fewshot", "lora") or l.startswith("Report")))
print(r.stderr[-400:] if r.returncode else "")

## 7. Discussion prompts (answer in your report)
1. LoRA trained only a small fraction of the parameters (check the printed percentage). What did we trade off versus full fine-tuning?
2. `nn_copy` needs no model at all. How close does it get - and what does that say about this templated dataset?
3. Which metric would you trust to decide whether the feedback actually *teaches* better? How would you collect it (human raters? student outcomes)?
4. What review process, bias audit and consent would you need before the model's feedback is shown to real students?